In [1]:
# --- Cell 0a: Phase 1 c-5 notebook, cells 1-2 ---
from collections import Counter
from tqdm.auto import tqdm
import pickle
import re
from collections import Counter, defaultdict

import os
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

print("Hugging Face login successful.")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Hugging Face login successful.


In [2]:
# --- Cell 0b: Phase 1 c-5 notebook, cell 8 (tokenizer) ---
SPECIAL_TOKENS = ["<pad>", "<unk>", "<s>", "</s>"]  # fixed ids 0-3, never merged

class BPETokenizer:
    def __init__(self):
        self.id_to_bytes = {}
        self.bytes_to_id = {}
        self.merges = {}
        self.merge_order = []
        self.merge_ranks = {}
        self.cache = {}
        self._vocab_size = 0
        self.split_pattern = re.compile(r'\s*\S+|\s+')

    def train(self, text: str, vocab_size: int):
        for i, tok in enumerate(SPECIAL_TOKENS):
            b = tok.encode("utf-8")
            self.id_to_bytes[i] = b
            self.bytes_to_id[b] = i
        next_id = len(SPECIAL_TOKENS)
        byte_to_base_id = {}
        for value in range(256):
            b = bytes([value])
            self.id_to_bytes[next_id] = b
            self.bytes_to_id[b] = next_id
            byte_to_base_id[value] = next_id
            next_id += 1
        chunks = self.split_pattern.findall(text)
        chunk_counts = Counter(chunks)
        chunk_to_ids = {chunk: tuple(byte_to_base_id[b] for b in chunk.encode("utf-8")) for chunk in chunk_counts}
        pair_counts = Counter()
        pair_to_chunks = defaultdict(set)
        def get_pairs(ids):
            return [(ids[i], ids[i+1]) for i in range(len(ids) - 1)]
        for chunk, ids in chunk_to_ids.items():
            count = chunk_counts[chunk]
            for pair in get_pairs(ids):
                pair_counts[pair] += count
                pair_to_chunks[pair].add(chunk)
        target = vocab_size
        pbar = tqdm(total=target - next_id, desc="Training BPE")
        while next_id < target:
            if not pair_counts:
                break
            max_freq = max(pair_counts.values())
            if max_freq <= 1:
                break
            best_pair = min(p for p, c in pair_counts.items() if c == max_freq)
            new_id = next_id
            self.merges[best_pair] = new_id
            self.merge_order.append(best_pair)
            self.merge_ranks[best_pair] = len(self.merge_order) - 1
            self.id_to_bytes[new_id] = self.id_to_bytes[best_pair[0]] + self.id_to_bytes[best_pair[1]]
            self.bytes_to_id[self.id_to_bytes[new_id]] = new_id
            for chunk in list(pair_to_chunks[best_pair]):
                old_ids = chunk_to_ids[chunk]
                count = chunk_counts[chunk]
                for pair in get_pairs(old_ids):
                    pair_counts[pair] -= count
                    if pair_counts[pair] <= 0:
                        del pair_counts[pair]
                    if chunk in pair_to_chunks[pair]:
                        pair_to_chunks[pair].remove(chunk)
                new_ids = []
                i = 0
                while i < len(old_ids):
                    if i < len(old_ids) - 1 and (old_ids[i], old_ids[i+1]) == best_pair:
                        new_ids.append(new_id); i += 2
                    else:
                        new_ids.append(old_ids[i]); i += 1
                new_ids = tuple(new_ids)
                chunk_to_ids[chunk] = new_ids
                for pair in get_pairs(new_ids):
                    pair_counts[pair] += count
                    pair_to_chunks[pair].add(chunk)
            next_id += 1
            pbar.update(1)
        pbar.close()
        self._vocab_size = next_id

    def encode(self, text: str):
        out_ids = []
        for chunk in self.split_pattern.findall(text):
            if chunk in self.cache:
                out_ids.extend(self.cache[chunk]); continue
            ids = [self.bytes_to_id[bytes([b])] for b in chunk.encode("utf-8")]
            while len(ids) > 1:
                pairs = [(ids[i], ids[i+1]) for i in range(len(ids) - 1)]
                best_pair, best_rank = None, float('inf')
                for p in pairs:
                    rank = self.merge_ranks.get(p, float('inf'))
                    if rank < best_rank:
                        best_rank, best_pair = rank, p
                if best_pair is None:
                    break
                new_id = self.merges[best_pair]
                new_ids, i = [], 0
                while i < len(ids):
                    if i < len(ids) - 1 and (ids[i], ids[i+1]) == best_pair:
                        new_ids.append(new_id); i += 2
                    else:
                        new_ids.append(ids[i]); i += 1
                ids = new_ids
            self.cache[chunk] = ids
            out_ids.extend(ids)
        return out_ids

    def decode(self, ids):
        return b"".join(self.id_to_bytes[i] for i in ids).decode("utf-8", errors="replace")

    def get_vocab_size(self):
        return self._vocab_size

    @property
    def vocab_size(self):
        return self._vocab_size


def train_tokenizer(train_text: str, vocab_size: int):
    tok = BPETokenizer()
    tok.train(train_text, vocab_size)
    return tok

In [3]:
# --- Cell 0c: Phase 1 c-5 notebook, cell 21 (model) ---
import math
from dataclasses import dataclass, asdict
import torch
import torch.nn as nn
import torch.nn.functional as F


class CausalSelfAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int, n_kv_heads: int, max_seq_len: int, dropout: float = 0.0, attention_scale=None):
        super().__init__()
        assert d_model % n_heads == 0
        assert n_heads % n_kv_heads == 0, "n_heads must be divisible by n_kv_heads"
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.n_rep = n_heads // n_kv_heads
        self.d_head = d_model // n_heads
        self.attention_scale = (1.0 / math.sqrt(self.d_head)) if attention_scale is None else float(attention_scale)
        self.q_proj = nn.Linear(d_model,n_heads * self.d_head)
        self.k_proj = nn.Linear(d_model,n_kv_heads * self.d_head)
        self.v_proj = nn.Linear(d_model,n_kv_heads * self.d_head)
        self.out_proj = nn.Linear(n_heads * self.d_head, d_model, bias=False)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)
        mask = torch.tril(torch.ones(max_seq_len, max_seq_len, dtype=torch.bool))
        self.register_buffer("causal_mask", mask, persistent=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1,2)
        k = self.k_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1,2)
        v = self.v_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1,2)
        if self.n_rep > 1:
            k = k.repeat_interleave(self.n_rep, dim=1)
            v = v.repeat_interleave(self.n_rep, dim=1)
        att = (q @ k.transpose(-2, -1)) * self.attention_scale
        att = att.masked_fill(self.causal_mask[:T, :T] == 0, float("-inf"))
        att = F.softmax(att, dim=-1)
        att = self.attn_dropout(att)
        y = att @ v
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.out_proj(y))


@dataclass
class LMConfig:
    vocab_size: int
    d_model: int = 1024
    n_layers: int = 6
    n_heads: int = 8
    n_kv_heads: int = 4
    d_ff: int = 1024
    max_seq_len: int = 256
    dropout: float = 0.1
    pos_emb_type: str = "sinusoidal"
    attention_scale: float = None
    norm_first: bool = False


def sinusoidal_positional_encoding(max_len: int, d_model: int) -> torch.Tensor:
    pe = torch.zeros(max_len, d_model)
    for a in range(max_len):
        for b in range(d_model):
            i = b // 2
            angle = a / (10000 ** (2 * i / d_model))
            if b % 2 == 0:
                pe[a][b] = math.sin(angle)
            else:
                pe[a][b] = math.cos(angle)
    return pe


class FeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model), nn.Dropout(dropout))

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.d_model)
        self.attn = CausalSelfAttention(cfg.d_model, cfg.n_heads, cfg.n_kv_heads, cfg.max_seq_len, cfg.dropout, attention_scale=cfg.attention_scale)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.ffn = FeedForward(cfg.d_model, cfg.d_ff, cfg.dropout)
        self.norm_first = cfg.norm_first

    def forward(self, x):
        if self.norm_first:
            x = x + self.attn(self.ln1(x))
            x = x + self.ffn(self.ln2(x))
        else:
            x = self.ln1(x + self.attn(x))
            x = self.ln2(x + self.ffn(x))
        return x


class DecoderLM(nn.Module):
    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        if cfg.pos_emb_type == "sinusoidal":
            self.register_buffer("pos_emb", sinusoidal_positional_encoding(cfg.max_seq_len, cfg.d_model), persistent=False)
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])
        self.ln_f = nn.LayerNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight  # weight tying

    def forward(self, idx: torch.Tensor) -> torch.Tensor:
        B, T = idx.shape
        x = self.tok_emb(idx)
        if self.cfg.pos_emb_type == "sinusoidal":
            x = x + self.pos_emb[:T, :]
        for block in self.blocks:
            x = block(x)
        return self.head(self.ln_f(x))

In [4]:
# CELL A — load the Phase 1 c-5 decoder and its tokenizer
import torch, pickle, json, re, os
ROOT = "/kaggle/input/notebooks/rudrajain2006/phase-1-c-5-simple-deepseek"   # change if your input folder differs
device = torch.device("cuda")

TOK_PATH  = next(p for p in (f"{ROOT}/cs781_phase1_full/bpe_tokenizer.pk", f"{ROOT}/cs781_phase1_full/legal-gpt2-final/bpe_tokenizer.pkl") if os.path.exists(p))
CKPT_PATH = next(p for p in (f"{ROOT}/cs781_phase1_full/legal-gpt2-final/ckpt.pt", f"{ROOT}/cs781_phase1_full/legal-gpt2-final/model_final.pt") if os.path.exists(p))
print("tokenizer:", TOK_PATH, "| checkpoint:", CKPT_PATH)

tok = pickle.load(open(TOK_PATH, "rb"))
ck = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)
cfg = LMConfig(**ck["config"])
lm = DecoderLM(cfg); print(lm.load_state_dict(ck["model"])); lm.to(device).eval()
del ck
print("params", sum(p.numel() for p in lm.parameters())/1e6, "M | vocab", tok.vocab_size,
      "| d_model", cfg.d_model, "| layers", cfg.n_layers, "| heads", cfg.n_heads, "| kv heads", cfg.n_kv_heads,
      "| d_ff", cfg.d_ff, "| max_seq_len", cfg.max_seq_len, "| norm_first", cfg.norm_first, "| dropout", cfg.dropout)

tokenizer: /kaggle/input/notebooks/rudrajain2006/phase-1-c-5-simple-deepseek/cs781_phase1_full/legal-gpt2-final/bpe_tokenizer.pkl | checkpoint: /kaggle/input/notebooks/rudrajain2006/phase-1-c-5-simple-deepseek/cs781_phase1_full/legal-gpt2-final/ckpt.pt
<All keys matched successfully>
params 98.408448 M | vocab 10000 | d_model 1024 | layers 8 | heads 8 | kv heads 2 | d_ff 4096 | max_seq_len 2048 | norm_first False | dropout 0.1


In [5]:
# CELL B — per-token vectors from the c-5 decoder (last hidden states, LM head skipped) + sanity check
def features(idx):                        # idx: (B, T) LongTensor, T <= 2048
    x = lm.tok_emb(idx)
    if lm.cfg.pos_emb_type == "sinusoidal":
        x = x + lm.pos_emb[:idx.size(1), :]
    for blk in lm.blocks:
        x = blk(x)
    return lm.ln_f(x)                     # (B, T, 1024)

idx = torch.randint(4, cfg.vocab_size, (1, 64), device=device)
with torch.no_grad():
    f = features(idx); diff = (lm.head(f) - lm(idx)).abs().max().item()
print("max diff", diff, tuple(f.shape))   # expect 0.0, (1, 64, 1024)

max diff 0.0 (1, 64, 1024)


In [6]:
# CELL C — Phase 3 (RR) data: load CL + IT train/dev/test and the label vocabulary
import json
from collections import Counter
from huggingface_hub import hf_hub_download

def get3(name):
    p = hf_hub_download("Exploration-Lab/CS-781-Capstone", f"Phase_3_RR/{name}", repo_type="dataset")
    return json.load(open(p, encoding="utf-8"))

_raw = get3("Train/CL_train.json")
print("train layout:", type(_raw).__name__, len(_raw))
_t = get3("Test/CL_test.json")
print("test layout:", type(_t).__name__, len(_t),
      str(_t[0] if isinstance(_t, list) else next(iter(_t.items())))[:300])

def as_docs(raw):
    """-> {doc_id: {"text": [sentences], "labels": [labels] or None}}"""
    while isinstance(raw, dict) and len(raw) == 1:
        v = next(iter(raw.values()))
        if (isinstance(v, list) and v and isinstance(v[0], dict)) or \
           (isinstance(v, dict) and v and all(isinstance(x, (dict, list)) for x in v.values())):
            raw = v
        else:
            break
    if isinstance(raw, dict) and "id" in raw and "text" in raw and isinstance(raw["id"], list):
        labs = raw.get("labels") or [None] * len(raw["id"])
        raw = [{"id": i, "text": t, "labels": l} for i, t, l in zip(raw["id"], raw["text"], labs)]
    if isinstance(raw, list):
        items = [(d.get("id", n), d) for n, d in enumerate(raw)]
    elif isinstance(raw, dict):
        items = list(raw.items())
    else:
        raise ValueError(f"unexpected file layout {type(raw).__name__}: print it and adapt as_docs")
    docs = {}
    for k, d in items:
        if isinstance(d, list):
            d = {"text": d}
        assert isinstance(d, dict), f"doc {k}: unexpected entry {type(d).__name__}"
        k = str(d.get("id", k))
        assert k not in docs, f"duplicate doc id {k}"
        text = d.get("text", d.get("sentences"))
        assert isinstance(text, list), f"doc {k}: no list of sentences under 'text' (keys {list(d)})"
        labs = d.get("labels")
        docs[k] = {"text": list(text), "labels": list(labs) if labs is not None else None}
    return docs

DOMAINS = ("CL", "IT")
train = {dom: as_docs(get3(f"Train/{dom}_train.json")) for dom in DOMAINS}
dev   = {dom: as_docs(get3(f"Train/{dom}_dev.json"))   for dom in DOMAINS}
test  = {dom: as_docs(get3(f"Test/{dom}_test.json"))   for dom in DOMAINS}
print("first test ids:", list(test["CL"])[:3])

lv = get3("Train/label_vocab.json")
if isinstance(lv, dict) and len(lv) == 1 and isinstance(next(iter(lv.values())), list):
    lv = next(iter(lv.values()))
if isinstance(lv, dict):
    if all(isinstance(v, int) for v in lv.values()):
        RR_LABELS = sorted(lv, key=lv.get)
    else:
        RR_LABELS = [lv[k] for k in sorted(lv, key=int)]
else:
    RR_LABELS = list(lv)
print(RR_LABELS)
assert len(RR_LABELS) == 13 and "None" in RR_LABELS, "check label_vocab.json by hand"
LAB_SET = set(RR_LABELS)

for split_name, split in (("train", train), ("dev", dev), ("test", test)):
    for dom in DOMAINS:
        docs = split[dom]
        n_sent = sum(len(d["text"]) for d in docs.values())
        print(f"{split_name:5s} {dom}: {len(docs):4d} docs, {n_sent:6d} sentences")
        for k, d in docs.items():
            if d["labels"] is None:
                continue
            assert len(d["text"]) == len(d["labels"]), f"{split_name}/{dom}/{k}: text vs labels length"
            bad = [l for l in d["labels"] if l not in LAB_SET]
            assert not bad, f"{split_name}/{dom}/{k}: labels not in vocab {set(map(repr, bad))}"

for split_name, split in (("train", train), ("dev", dev)):
    for dom in DOMAINS:
        c = Counter(l for d in split[dom].values() for l in d["labels"])
        print(split_name, dom, {l: c.get(l, 0) for l in RR_LABELS})

for split_name, split in (("train", train), ("dev", dev), ("test", test)):
    print(f"{split_name}: CL∩IT ids = {len(set(split['CL']) & set(split['IT']))}")
for dom in DOMAINS:
    print(f"{dom}: train∩dev ids = {len(set(train[dom]) & set(dev[dom]))}")

CL_train.json:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

train layout: list 40


CL_test.json:   0%|          | 0.00/281k [00:00<?, ?B/s]

test layout: dict 5 ('HC_Raghupati_Singhania__vs__Competition_Commission_ofDE201616051616063761COM513489', ['Manmohan, J. 1 .', 'Present writ petition has been filed, challenging the notice dated 26th February, 2016 and respondent Number 1s order dated 18th February, 2016 whereby the respondent-CCI has directed the opp


IT_train.json:   0%|          | 0.00/1.51M [00:00<?, ?B/s]

CL_dev.json:   0%|          | 0.00/336k [00:00<?, ?B/s]

IT_dev.json:   0%|          | 0.00/165k [00:00<?, ?B/s]

IT_test.json:   0%|          | 0.00/194k [00:00<?, ?B/s]

first test ids: ['HC_Raghupati_Singhania__vs__Competition_Commission_ofDE201616051616063761COM513489', 'COM_TPM_Consultants_Pvt_Ltd_vs_Competition_Commission_TA2016110716153210127COM903224', 'SC_Ultra Tech Cement Ltd vs Competition Commission of India and Ors 12062013 SC']


label_vocab.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

['Fact', 'Issue', 'ArgumentPetitioner', 'ArgumentRespondent', 'Statute', 'Dissent', 'PrecedentReliedUpon', 'PrecedentNotReliedUpon', 'PrecedentOverruled', 'RulingByLowerCourt', 'RatioOfTheDecision', 'RulingByPresentCourt', 'None']
train CL:   40 docs,  10593 sentences
train IT:   40 docs,   6252 sentences
dev   CL:    5 docs,   1396 sentences
dev   IT:    5 docs,    746 sentences
test  CL:    5 docs,   1339 sentences
test  IT:    5 docs,    858 sentences
train CL {'Fact': 2979, 'Issue': 17, 'ArgumentPetitioner': 976, 'ArgumentRespondent': 996, 'Statute': 367, 'Dissent': 284, 'PrecedentReliedUpon': 1305, 'PrecedentNotReliedUpon': 209, 'PrecedentOverruled': 0, 'RulingByLowerCourt': 438, 'RatioOfTheDecision': 2856, 'RulingByPresentCourt': 113, 'None': 53}
train IT {'Fact': 1119, 'Issue': 90, 'ArgumentPetitioner': 637, 'ArgumentRespondent': 352, 'Statute': 732, 'Dissent': 0, 'PrecedentReliedUpon': 640, 'PrecedentNotReliedUpon': 360, 'PrecedentOverruled': 3, 'RulingByLowerCourt': 436, 'Rati

In [7]:
# CELL D — sentence tokenisation, local scorer, prediction writer, majority baseline
import json, os, zipfile
import numpy as np
from collections import Counter
from tqdm.auto import tqdm

UNK_ID = SPECIAL_TOKENS.index("<unk>")   # 1
EOS_ID = SPECIAL_TOKENS.index("</s>")    # 3

PREFIX_SPACE = False   # <-- PICK True/False: encode " " + sentence?
SEP_TOKEN    = "</s>"   # <-- PICK "none" or "</s>": what goes BETWEEN sentences in a packed window
assert PREFIX_SPACE is not None and SEP_TOKEN in ("none", "</s>"), "set PREFIX_SPACE and SEP_TOKEN"
SEP_ID = None if SEP_TOKEN == "none" else EOS_ID

def encode_sentence(s):
    ids = tok.encode((" " + s) if PREFIX_SPACE else s)
    return ids if ids else [UNK_ID]

for split in (train, dev, test):
    for dom in DOMAINS:
        for d in tqdm(split[dom].values(), desc="tokenising sentences", leave=False):
            d["ids"] = [encode_sentence(s) for s in d["text"]]

for dom in DOMAINS:   # lengths, to choose MAX_SENT_TOKENS / PACK_TOKENS (<= 2048 for c-5)
    sl = np.array([len(x) for d in train[dom].values() for x in d["ids"]])
    dl = np.array([sum(map(len, d["ids"])) for d in train[dom].values()])
    print(f"{dom} sentence tokens: median {np.median(sl):.0f} p95 {np.percentile(sl, 95):.0f} max {sl.max()} | "
          f"doc tokens: median {np.median(dl):.0f} p95 {np.percentile(dl, 95):.0f} max {dl.max()}")


def macro_scores(gold_lists, pred_lists):
    """macro-F1 over the labels that appear in gold or predictions (sklearn 'macro')"""
    tp, fp, fn = Counter(), Counter(), Counter()
    for g, p in zip(gold_lists, pred_lists):
        assert len(g) == len(p), "one prediction per sentence"
        for a, b in zip(g, p):
            if a == b: tp[a] += 1
            else:      fp[b] += 1; fn[a] += 1
    used = [l for l in RR_LABELS if tp[l] + fp[l] + fn[l] > 0]
    per = {}
    for l in used:
        P = tp[l] / (tp[l] + fp[l]) if tp[l] + fp[l] else 0.0
        R = tp[l] / (tp[l] + fn[l]) if tp[l] + fn[l] else 0.0
        per[l] = (P, R, 2 * P * R / (P + R) if P + R else 0.0, tp[l] + fn[l])
    m = lambda i: sum(v[i] for v in per.values()) / max(1, len(per))
    return {"p": m(0), "r": m(1), "f1": m(2), "per": per}

def evaluate(pred_by_domain, docs_by_domain, show=True):
    res, allg, allp = {}, [], []
    for dom, docs in docs_by_domain.items():
        g = [d["labels"] for d in docs.values()]
        p = [pred_by_domain[dom][k] for k in docs]
        res[dom] = macro_scores(g, p); allg += g; allp += p
    res["ALL"] = macro_scores(allg, allp)
    if show:
        for k, v in res.items():
            print(f"{k:3s} macro-F1 {v['f1']:.4f}  P {v['p']:.4f}  R {v['r']:.4f}")
    return res

def per_label_table(res):
    for l in RR_LABELS:
        row = [f"{res[k]['per'][l][2]:.3f} (n={res[k]['per'][l][3]})" if l in res[k]["per"] else "  -  "
               for k in ("CL", "IT")]
        print(f"  {l:24s} CL F1 {row[0]:16s} IT F1 {row[1]}")

def confusion(pred_by_domain, docs_by_domain, dom):
    idx = {l: i for i, l in enumerate(RR_LABELS)}
    M = np.zeros((len(RR_LABELS), len(RR_LABELS)), dtype=int)
    for k, d in docs_by_domain[dom].items():
        for g, p in zip(d["labels"], pred_by_domain[dom][k]):
            M[idx[g], idx[p]] += 1
    short = [l[:6] for l in RR_LABELS]
    print(f"{dom} confusion (rows gold, cols pred)\n" + " " * 8 + " ".join(f"{s:>6s}" for s in short))
    for s, row in zip(short, M):
        print(f"{s:>7s} " + " ".join(f"{v:6d}" for v in row))
    return M

def write_prediction(pred_by_domain, test_docs, out_dir="/kaggle/working"):
    out = {}
    for dom, docs in test_docs.items():
        for doc_id, d in docs.items():
            labs = pred_by_domain[dom][doc_id]
            assert len(labs) == len(d["text"]), f"{dom}/{doc_id}: wrong number of predictions"
            assert all(isinstance(l, str) and l in LAB_SET for l in labs), f"{dom}/{doc_id}: bad label"
            assert doc_id not in out, f"doc id {doc_id} is in both CL and IT test files: ask the TAs"
            out[doc_id] = list(labs)
    assert len(out) == sum(len(v) for v in test_docs.values()), "every test doc ID, no extras"
    path = os.path.join(out_dir, "prediction.json")
    with open(path, "w", encoding="utf-8") as f:
        json.dump(out, f, ensure_ascii=False)
    zpath = os.path.join(out_dir, "submission.zip")
    with zipfile.ZipFile(zpath, "w", compression=zipfile.ZIP_DEFLATED) as z:
        z.write(path, arcname="prediction.json")
    with zipfile.ZipFile(zpath) as z:
        assert z.namelist() == ["prediction.json"], z.namelist()
    print(f"{len(out)} docs, {sum(map(len, out.values()))} sentences -> {zpath}")

# Majority baseline: checks scorer + writer end to end (don't upload it)
majority = {dom: Counter(l for d in train[dom].values() for l in d["labels"]).most_common(1)[0][0] for dom in DOMAINS}
print("majority label per domain:", majority)
evaluate({dom: {k: [majority[dom]] * len(d["text"]) for k, d in dev[dom].items()} for dom in DOMAINS}, dev)
write_prediction({dom: {k: [majority[dom]] * len(d["text"]) for k, d in test[dom].items()} for dom in DOMAINS}, test)

tokenising sentences:   0%|          | 0/40 [00:00<?, ?it/s]

tokenising sentences:   0%|          | 0/40 [00:00<?, ?it/s]

tokenising sentences:   0%|          | 0/5 [00:00<?, ?it/s]

tokenising sentences:   0%|          | 0/5 [00:00<?, ?it/s]

tokenising sentences:   0%|          | 0/5 [00:00<?, ?it/s]

tokenising sentences:   0%|          | 0/5 [00:00<?, ?it/s]

CL sentence tokens: median 37 p95 97 max 1116 | doc tokens: median 5796 p95 48127 max 83276
IT sentence tokens: median 36 p95 113 max 1363 | doc tokens: median 6079 p95 17740 max 22313
majority label per domain: {'CL': 'Fact', 'IT': 'RatioOfTheDecision'}
CL  macro-F1 0.0303  P 0.0178  R 0.1000
IT  macro-F1 0.0243  P 0.0140  R 0.0909
ALL macro-F1 0.0428  P 0.0302  R 0.0769
10 docs, 2197 sentences -> /kaggle/working/submission.zip


In [8]:
# CELL E0a — shared plumbing for the c-5 Phase 3 (RR) experiments (part 1)
import math, time, random
from collections import defaultdict
import torch, torch.nn as nn, torch.nn.functional as F
from tqdm.auto import tqdm

N_LABELS = len(RR_LABELS)
LAB2ID = {l: i for i, l in enumerate(RR_LABELS)}
IGNORE = -100            # sentences with no gold label (test) or read-only context
FEAT_DIM = cfg.d_model   # 1024
PAD_ID = SPECIAL_TOKENS.index("<pad>")   # 0; c-5 is causal, so right padding is safe


def decoder_forward(backbone, idx):
    T = idx.size(1)
    assert T <= backbone.cfg.max_seq_len, "sequence longer than 2048"
    x = backbone.tok_emb(idx)
    if backbone.cfg.pos_emb_type == "sinusoidal":
        x = x + backbone.pos_emb[:T, :]
    for blk in backbone.blocks:
        x = blk(x)
    return backbone.ln_f(x)                      # (B, T, 1024); LM head skipped


def decoder_linears(backbone, which):
    """Yield (parent, attr, nn.Linear) for the chosen linears in every decoder block.
    which ⊆ {"q_proj", "k_proj", "v_proj", "out_proj", "ffn_in", "ffn_out"}.
    c-5 GQA: q_proj 1024->1024, k_proj/v_proj 1024->256, all with bias; out_proj 1024->1024 no bias;
    ffn_in 1024->4096, ffn_out 4096->1024."""
    for blk in backbone.blocks:
        for w in which:
            if w in ("q_proj", "k_proj", "v_proj", "out_proj"):
                parent, attr = blk.attn, w
            elif w == "ffn_in":
                parent, attr = blk.ffn.net, "0"
            elif w == "ffn_out":
                parent, attr = blk.ffn.net, "2"
            else:
                raise ValueError(f"{w}: c-5 targets are q_proj, k_proj, v_proj, out_proj, ffn_in, ffn_out")
            yield parent, attr, getattr(parent, attr)


def wrap_linears(backbone, which, make_wrapper):
    todo = list(decoder_linears(backbone, which))
    for parent, attr, lin in todo:
        assert isinstance(lin, nn.Linear), f"{attr} is already wrapped (re-run Cell A to start fresh)"
        setattr(parent, attr, make_wrapper(lin))
    return len(todo)


class RRModel(nn.Module):
    """Sentence-mode model (one sentence per head call). The context run uses SeqRRModel (E0c1)."""
    def __init__(self, backbone, pool, head):
        super().__init__()
        self.backbone, self.pool, self.head = backbone, pool, head
    def forward(self, ids, spans):
        h = decoder_forward(self.backbone, ids)
        vecs = [self.pool(h[b, s:e]) for b, sp in enumerate(spans) for (s, e) in sp]
        return self.head(torch.stack(vecs))


def identity_probe():
    d = next(iter(train["CL"].values()))
    ids = [t for x in d["ids"] for t in x][:64]
    return torch.tensor([ids], device=device)


def count_params(model):
    tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    tot = sum(p.numel() for p in model.parameters())
    print(f"trainable {tr/1e6:.3f}M / total {tot/1e6:.2f}M ({100*tr/tot:.2f}%)")
    return tr, tot


def make_windows(n, window, stride):
    if n <= window:
        return [(0, n)]
    starts = list(range(0, n - window, stride)) + [n - window]
    return [(s, s + window) for s in starts]


def build_examples(docs_by_domain, mode, *, max_sent_tokens=None, window=None, stride=None):
    """Sentence/document-mode examples (the context run uses build_packed_examples in E0c1)."""
    assert mode in ("sentence", "document"), mode
    examples = []
    for dom, docs in docs_by_domain.items():
        for doc_id, d in tqdm(docs.items(), desc=f"{dom} examples ({mode})", leave=False):
            labs = [LAB2ID[l] for l in d["labels"]] if d["labels"] is not None else [IGNORE] * len(d["ids"])
            if mode == "sentence":
                for i, ids in enumerate(d["ids"]):
                    ids = ids[:max_sent_tokens]
                    examples.append({"ids": ids, "spans": [(0, len(ids))], "keys": [(dom, doc_id, i)], "labels": [labs[i]]})
                continue
            stream, sp = [], []
            for i, ids in enumerate(d["ids"]):
                if i > 0 and SEP_ID is not None:
                    stream.append(SEP_ID)
                sp.append((len(stream), len(stream) + len(ids)))
                stream += ids
            if not stream:
                continue
            wins = make_windows(len(stream), window, stride)
            owner = []
            for s, e in sp:
                def score(w):
                    ws, we = wins[w]
                    if ws <= s and e <= we:
                        return s - ws
                    return min(e, we) - max(s, ws) - 10 ** 9
                owner.append(max(range(len(wins)), key=score))
            for w, (ws, we) in enumerate(wins):
                own = [i for i in range(len(sp)) if owner[i] == w]
                if own:
                    examples.append({"ids": stream[ws:we],
                                     "spans": [(max(sp[i][0], ws) - ws, min(sp[i][1], we) - ws) for i in own],
                                     "keys": [(dom, doc_id, i) for i in own], "labels": [labs[i] for i in own]})
    return examples

In [9]:
# CELL E0b — shared plumbing for the c-5 Phase 3 (RR) experiments (part 2)
def make_batches(examples, batch_size, shuffle):
    """Batches of similar length, RIGHT-padded in to_tensors (safe for a causal decoder; pads never pooled)."""
    exs = list(examples)
    if shuffle:
        random.shuffle(exs)
    out, pool = [], batch_size * 50
    for i in range(0, len(exs), pool):
        chunk = sorted(exs[i:i + pool], key=lambda ex: len(ex["ids"]))
        out += [chunk[j:j + batch_size] for j in range(0, len(chunk), batch_size)]
    if shuffle:
        random.shuffle(out)
    return out


def to_tensors(batch):
    T = max(len(ex["ids"]) for ex in batch)
    ids = torch.full((len(batch), T), PAD_ID, dtype=torch.long)
    for i, ex in enumerate(batch):
        ids[i, :len(ex["ids"])] = torch.tensor(ex["ids"])
    return {"ids": ids.to(device),
            "spans": [ex["spans"] for ex in batch],
            "labels": torch.tensor([l for ex in batch for l in ex["labels"]], device=device)}


def ce_loss(logits, labels, class_weights=None):
    """Sentence-level cross-entropy; IGNORE rows (context sentences) don't count."""
    return F.cross_entropy(logits.float(), labels, weight=class_weights, ignore_index=IGNORE)


@torch.no_grad()
def time_inference(model, examples, docs_by_domain, batch_size):
    """Wall-clock seconds to predict a set (for the efficiency table in the report)."""
    torch.cuda.synchronize(); t = time.time()
    predict(model, examples, docs_by_domain, batch_size)
    torch.cuda.synchronize(); return time.time() - t

In [10]:
# CELL E0c1 — document-context plumbing for c-5, part 1 of 2
#   packed window = [up to ctx_left earlier sentences, read-only] + [as many next sentences
#   as fit in pack_tokens]. Context sentences get label IGNORE and key None: pooled and shown
#   to the head, never scored or written. Every sentence is predicted exactly once.

def build_packed_examples(docs_by_domain, *, pack_tokens, ctx_left, max_sent_tokens):
    assert pack_tokens <= cfg.max_seq_len, f"pack_tokens > max_seq_len {cfg.max_seq_len}"
    sep = 1 if SEP_ID is not None else 0
    examples = []
    for dom, docs in docs_by_domain.items():
        for doc_id, d in tqdm(docs.items(), desc=f"{dom} examples (packed)", leave=False):
            sents = [ids[:max_sent_tokens] for ids in d["ids"]]
            n = len(sents)
            labs = [LAB2ID[l] for l in d["labels"]] if d["labels"] is not None else [IGNORE] * n
            cost = lambda i: len(sents[i]) + sep
            s = 0
            while s < n:
                c0 = max(0, s - ctx_left)                       # left context: <= ctx_left sentences
                used = sum(cost(i) for i in range(c0, s))
                while c0 < s and used > pack_tokens // 2:       # ... and <= half the budget
                    used -= cost(c0); c0 += 1
                e = s
                while e < n and (e == s or used + cost(e) <= pack_tokens):
                    used += cost(e); e += 1
                ids, spans = [], []
                for i in range(c0, e):
                    if i > c0 and SEP_ID is not None:
                        ids.append(SEP_ID)
                    spans.append((len(ids), len(ids) + len(sents[i])))
                    ids += sents[i]
                assert len(ids) <= cfg.max_seq_len, "window longer than max_seq_len: lower pack_tokens"
                examples.append({
                    "ids": ids, "spans": spans,
                    "keys": [(dom, doc_id, i) if i >= s else None for i in range(c0, e)],
                    "labels": [labs[i] if i >= s else IGNORE for i in range(c0, e)]})
                s = e
    return examples


class SeqRRModel(nn.Module):
    """decoder -> pool each sentence -> (B, S, pooled_dim) padded per window -> head(Z, lengths)
    -> (B, S, 13) -> one logit row per sentence, in span order."""
    def __init__(self, backbone, pool, head):
        super().__init__()
        self.backbone, self.pool, self.head = backbone, pool, head
    def forward(self, ids, spans):
        h = decoder_forward(self.backbone, ids)                                     # (B, T, 1024)
        per_win = [torch.stack([self.pool(h[b, s:e]) for (s, e) in sp]) for b, sp in enumerate(spans)]
        lengths = torch.tensor([len(sp) for sp in spans])
        Z = nn.utils.rnn.pad_sequence(per_win, batch_first=True)                    # (B, S, pooled_dim)
        out = self.head(Z, lengths)                                                 # (B, S, 13)
        return torch.cat([out[b, :len(sp)] for b, sp in enumerate(spans)])          # (sum S, 13)


@torch.no_grad()
def predict(model, examples, docs_by_domain, batch_size):
    model.eval()
    out = {dom: {k: [None] * len(d["text"]) for k, d in docs.items()} for dom, docs in docs_by_domain.items()}
    for batch in tqdm(make_batches(examples, batch_size, shuffle=False), desc="predict", leave=False):
        b = to_tensors(batch)
        with torch.autocast("cuda", dtype=torch.float16):
            pred = model(b["ids"], b["spans"]).argmax(-1).tolist()
        keys = [k for ex in batch for k in ex["keys"]]
        for key, p in zip(keys, pred):
            if key is None:
                continue
            dom, doc_id, i = key
            assert out[dom][doc_id][i] is None, f"{dom}/{doc_id}/{i} predicted twice"
            out[dom][doc_id][i] = RR_LABELS[p]
    missing = sum(x is None for dom in out for v in out[dom].values() for x in v)
    assert missing == 0, f"{missing} sentences got no prediction"
    return out

In [11]:
# CELL E0c2 — document-context plumbing for c-5, part 2 of 2
def warmup_cosine(total_steps, warmup_frac):
    """LR multiplier per step: linear warm-up over warmup_frac of the steps, then cosine to 0."""
    warm = max(1, int(warmup_frac * total_steps))
    def f(step):
        if step < warm:
            return (step + 1) / warm
        return 0.5 * (1 + math.cos(math.pi * (step - warm) / max(1, total_steps - warm)))
    return f


def train_model(model, train_ex, dev_ex, dev_docs, loss_fn, *, lr, epochs, batch_size,
          weight_decay=0.0, grad_clip=1.0, lr_lambda=None, save_path="best_trainable.pt",
          param_groups=None, hooks=None):
    params = [p for p in model.parameters() if p.requires_grad]
    if param_groups is None:
        param_groups = [{"params": params, "lr": lr}]
    grouped = {id(p) for g in param_groups for p in g["params"]}
    assert grouped == {id(p) for p in params}, "param_groups must cover every trainable parameter exactly"
    opt = torch.optim.AdamW(param_groups, lr=lr, weight_decay=weight_decay)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda) if lr_lambda else None
    scaler = torch.amp.GradScaler("cuda")
    torch.cuda.reset_peak_memory_stats()
    t0, best_f1, best_res, log, skipped = time.time(), -1.0, None, [], 0
    for epoch in range(epochs):
        model.train(); tot, nb = 0.0, 0
        pbar = tqdm(make_batches(train_ex, batch_size, shuffle=True), desc=f"epoch {epoch + 1}/{epochs}")
        for batch in pbar:
            b = to_tensors(batch)
            with torch.autocast("cuda", dtype=torch.float16):
                logits = model(b["ids"], b["spans"])
            loss = loss_fn(logits.float(), b)
            opt.zero_grad(set_to_none=True)
            if not torch.isfinite(loss):
                skipped += 1
                if sched: sched.step()
                continue
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            if hooks: hooks.after_backward()                 # AdaLoRA: score importance (real gradients)
            torch.nn.utils.clip_grad_norm_(params, grad_clip)
            scaler.step(opt); scaler.update()
            if hooks: hooks.after_step()                     # AdaLoRA: prune ranks
            if sched: sched.step()
            tot += loss.item(); nb += 1
            pbar.set_postfix(loss=f"{tot / nb:.4f}", lr=f"{opt.param_groups[0]['lr']:.2e}", skipped=skipped,
                             **(hooks.postfix() if hooks else {}))
        res = evaluate(predict(model, dev_ex, dev_docs, batch_size), dev_docs, show=False)
        log.append({"epoch": epoch + 1, "train_loss": tot / max(1, nb), "dev_f1_ALL": res["ALL"]["f1"],
                    "dev_f1_CL": res["CL"]["f1"], "dev_f1_IT": res["IT"]["f1"], "skipped": skipped})
        if hooks: log[-1].update(hooks.postfix())
        print(log[-1])
        if res["ALL"]["f1"] > best_f1:
            best_f1, best_res = res["ALL"]["f1"], res
            torch.save({n: p.detach().cpu() for n, p in model.named_parameters() if p.requires_grad}, save_path)
    print(f"best dev macro-F1 ALL {best_f1:.4f} (CL {best_res['CL']['f1']:.4f}, IT {best_res['IT']['f1']:.4f}) | "
          f"train time {(time.time() - t0) / 60:.1f} min | peak GPU mem {torch.cuda.max_memory_allocated() / 2**30:.2f} GB")
    per_label_table(best_res)
    return log, best_res


def write_submission_packed(model, pack_kwargs, batch_size, best_path=None, out_dir="/kaggle/working"):
    if best_path:
        model.load_state_dict(torch.load(best_path), strict=False)
    test_ex = build_packed_examples(test, **pack_kwargs)
    write_prediction(predict(model, test_ex, test, batch_size), test, out_dir=out_dir)

In [12]:
# CELL E0d — AdaLoRA rank allocator for c-5 (follows PEFT's RankAllocator).
# Needs layers exposing ada_P, ada_E, ada_Q, ada_mask and orth_penalty().
class AdaLoRAAllocator:
    def __init__(self, layers, init_rank, target_rank, total_steps, tinit, tfinal, delta_t, beta1, beta2):
        assert tinit < total_steps - tfinal, "warm-up and final phase leave no steps for pruning"
        self.layers = layers
        self.init_budget, self.target_budget = init_rank * len(layers), target_rank * len(layers)
        self.T, self.tinit, self.tfinal, self.dT = total_steps, tinit, tfinal, delta_t
        self.b1, self.b2 = beta1, beta2
        self.ipt, self.unc = {}, {}              # smoothed importance and uncertainty per parameter
        self.step, self.budget = 0, self.init_budget

    def _params(self, m):
        return {"P": m.ada_P, "E": m.ada_E, "Q": m.ada_Q}

    @torch.no_grad()
    def after_backward(self):                    # update scores while ranks can still change
        if self.step >= self.T - self.tfinal:
            return
        for m in self.layers:
            for p in self._params(m).values():
                if p.grad is None or not torch.isfinite(p.grad).all():
                    return                       # fp16 overflow step: GradScaler skips it, so do we
        for li, m in enumerate(self.layers):
            for k, p in self._params(m).items():
                s = (p * p.grad).abs().float()
                key = (li, k)
                if key not in self.ipt:
                    self.ipt[key], self.unc[key] = torch.zeros_like(s), torch.zeros_like(s)
                self.ipt[key].mul_(self.b1).add_((1 - self.b1) * s)
                self.unc[key].mul_(self.b2).add_((1 - self.b2) * (s - self.ipt[key]).abs())

    def _triplet_scores(self, li):
        sc = {k: self.ipt[(li, k)] * self.unc[(li, k)] for k in ("P", "E", "Q")}
        return sc["E"] + sc["P"].mean(0) + sc["Q"].mean(1)   # (rank,)

    def _schedule(self):
        t = self.step
        if t <= self.tinit:
            return self.init_budget, False
        if t > self.T - self.tfinal:
            return self.target_budget, True
        mul = 1 - (t - self.tinit) / (self.T - self.tfinal - self.tinit)
        b = int((self.init_budget - self.target_budget) * mul ** 3 + self.target_budget)
        return b, t % self.dT == 0

    @torch.no_grad()
    def after_step(self):
        self.step += 1
        self.budget, do_mask = self._schedule()
        if not do_mask or not self.ipt:
            return
        if not getattr(self, "_frozen", False):
            # keep the `budget` highest-scoring triplets over ALL layers, prune the rest
            scores = [self._triplet_scores(li) for li in range(len(self.layers))]
            flat = torch.cat(scores)
            keep = torch.zeros_like(flat, dtype=torch.bool)
            keep[flat.topk(min(self.budget, flat.numel())).indices] = True
            for m, kp in zip(self.layers, keep.split([s.numel() for s in scores])):
                m.ada_mask.copy_(kp.float())
            if self.step > self.T - self.tfinal:
                self._frozen = True              # final phase: keep this rank pattern to the end
        for m in self.layers:
            m.ada_E.mul_(m.ada_mask)             # pruned triplets: E = 0 (P, Q kept, so they can revive)

    def postfix(self):
        return {"rank_budget": self.budget}

    def orth_penalty(self):
        return sum(m.orth_penalty() for m in self.layers) / len(self.layers)

    def rank_pattern(self):
        """Ranks actually used per layer = non-zero E entries (works after loading saved weights)."""
        return [int((m.ada_E != 0).sum()) for m in self.layers]

In [13]:

# CELL E2a-ctx — AdaLoRA on c-5 with DOCUMENT CONTEXT (part 1 of 2)
# Re-run Cell A and E0a..E0d first if starting from scratch.

import random
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F


# ============================================================
# [H] Settings and hyperparameters
# ============================================================
CONTEXT_MODE    = "packed"
MAX_SENT_TOKENS = 128
PACK_TOKENS     = 1024
CTX_LEFT        = 2
TRAIN_DOMAINS   = ("CL", "IT")

ADA_INIT_RANK   = 80
ADA_TARGETS     = ("q_proj", "k_proj", "v_proj", "out_proj")

LR              = 3e-5
HEAD_LR         = 3e-4
WARMUP_FRAC     = 0.10
EPOCHS          = 50
BATCH_SIZE      = 1
WEIGHT_DECAY    = 0.0
SEED            = 42

# AdaLoRA schedule
ADA_TARGET_RANK = round(ADA_INIT_RANK / 1.5)
TINIT_FRAC      = 0.10
TFINAL_FRAC     = 0.20
DELTA_T         = 10
BETA1, BETA2    = 0.85, 0.85
ORTH_REG        = 0.1

for _n in [
    "MAX_SENT_TOKENS",
    "PACK_TOKENS",
    "CTX_LEFT",
    "HEAD_LR",
    "WARMUP_FRAC",
    "BATCH_SIZE",
]:
    assert globals()[_n] is not None, f"set {_n}"

assert MAX_SENT_TOKENS <= PACK_TOKENS // 2, (
    "a context-capped window must still fit one full sentence"
)
assert 0 < ADA_TARGET_RANK <= ADA_INIT_RANK
assert TINIT_FRAC + TFINAL_FRAC < 1

PACK = {
    "pack_tokens": PACK_TOKENS,
    "ctx_left": CTX_LEFT,
    "max_sent_tokens": MAX_SENT_TOKENS,
}

random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

RUN_NAME = "c5_adalora_packed"


# ============================================================
# Class weights from training label counts
# ============================================================
WEIGHT_SCHEME = "sqrt_inverse"
assert WEIGHT_SCHEME in ("inverse", "sqrt_inverse", "none")

_cnt = Counter(
    label
    for domain in TRAIN_DOMAINS
    for doc in train[domain].values()
    for label in doc["labels"]
)

_counts = torch.tensor(
    [_cnt.get(label, 0) for label in RR_LABELS],
    dtype=torch.float,
)

_present = _counts > 0

if WEIGHT_SCHEME == "none":
    CLASS_WEIGHTS = None
else:
    _w = torch.zeros(len(RR_LABELS), dtype=torch.float)

    if WEIGHT_SCHEME == "inverse":
        _w[_present] = 1.0 / _counts[_present]
    else:
        _w[_present] = 1.0 / _counts[_present].sqrt()

    _w[_present] /= _w[_present].mean()
    CLASS_WEIGHTS = _w.to(device)

    for label, count, weight in zip(
        RR_LABELS, _counts.tolist(), _w.tolist()
    ):
        print(
            f"  {label:24s} train n={int(count):6d} "
            f"weight={weight:.3f}"
        )


# ============================================================
# [L] AdaLoRA layer
#
# Output = base(x) + P @ diag(E * mask) @ Q @ x
#
# P: (out_features, rank)
# E: (rank,)
# Q: (rank, in_features)
# ============================================================
class AdaLoRALinear(nn.Module):
    def __init__(self, base: nn.Linear, rank: int):
        super().__init__()

        self.base = base

        for p in self.base.parameters():
            p.requires_grad = False

        self.in_features = base.in_features
        self.out_features = base.out_features
        self.rank = rank

        self.ada_P = nn.Parameter(
            torch.empty(self.out_features, rank)
        )
        self.ada_E = nn.Parameter(
            torch.zeros(rank)
        )
        self.ada_Q = nn.Parameter(
            torch.empty(rank, self.in_features)
        )

        # Initialize P and Q with orthonormal columns/rows.
        nn.init.orthogonal_(self.ada_P)
        nn.init.orthogonal_(self.ada_Q)

        # The allocator can prune triplets by changing the mask
        # and/or setting the corresponding E entries to zero.
        self.register_buffer(
            "ada_mask",
            torch.ones(rank)
        )

    def forward(self, x):
        base_out = self.base(x)

        # Compute the adapter update in the activation dtype.
        P = self.ada_P.to(dtype=x.dtype)
        Q = self.ada_Q.to(dtype=x.dtype)
        E = self.ada_E.to(dtype=x.dtype)
        mask = self.ada_mask.to(dtype=x.dtype)

        # Q @ x, followed by elementwise singular-value scaling,
        # followed by P.
        qx = F.linear(x, Q)
        qx = qx * (E * mask)
        delta = F.linear(qx, P)

        return base_out + delta.to(dtype=base_out.dtype)

    def orth_penalty(self):
        # Encourage P.T @ P = I and Q @ Q.T = I.
        # Float32 is used for numerical stability.
        P = self.ada_P.float()
        Q = self.ada_Q.float()

        eye = torch.eye(
            self.rank,
            device=P.device,
            dtype=torch.float32,
        )

        gram_P = P.T @ P
        gram_Q = Q @ Q.T

        penalty_P = (gram_P - eye).pow(2).sum()
        penalty_Q = (gram_Q - eye).pow(2).sum()

        return penalty_P + penalty_Q


  Fact                     train n=  4098 weight=0.192
  Issue                    train n=   107 weight=1.186
  ArgumentPetitioner       train n=  1613 weight=0.305
  ArgumentRespondent       train n=  1348 weight=0.334
  Statute                  train n=  1099 weight=0.370
  Dissent                  train n=   284 weight=0.728
  PrecedentReliedUpon      train n=  1945 weight=0.278
  PrecedentNotReliedUpon   train n=   569 weight=0.514
  PrecedentOverruled       train n=     3 weight=7.080
  RulingByLowerCourt       train n=   874 weight=0.415
  RatioOfTheDecision       train n=  4301 weight=0.187
  RulingByPresentCourt     train n=   307 weight=0.700
  None                     train n=   297 weight=0.712


In [14]:

# CELL E2b-ctx — AdaLoRA on c-5 with DOCUMENT CONTEXT (part 2 of 2)

# ============================================================
# Freeze c-5, wrap target linears and check identity at init
# ============================================================
lm.eval()

probe = identity_probe()

with torch.no_grad():
    ref = features(probe).float()

for p in lm.parameters():
    p.requires_grad = False

n_wrapped = wrap_linears(
    lm,
    ADA_TARGETS,
    lambda lin: AdaLoRALinear(
        lin, ADA_INIT_RANK
    ).to(device),
)

ada_layers = [
    m for _, _, m in decoder_linears(lm, ADA_TARGETS)
]

ada_names = [
    f"block{i}.{w}"
    for i in range(cfg.n_layers)
    for w in ADA_TARGETS
]

for m in ada_layers:
    assert tuple(m.ada_P.shape) == (
        m.out_features, m.rank
    )
    assert tuple(m.ada_E.shape) == (m.rank,)
    assert tuple(m.ada_Q.shape) == (
        m.rank, m.in_features
    )
    assert tuple(m.ada_mask.shape) == (m.rank,)

with torch.no_grad():
    diff = (
        features(probe).float() - ref
    ).abs().max().item()

print(
    f"wrapped {n_wrapped} linears | "
    f"AdaLoRA identity-at-init max |diff| = {diff:.2e} "
    f"(should be ~0)"
)

assert diff < 1e-3, (
    "AdaLoRA layer is not the identity at init: "
    "check AdaLoRALinear initialization"
)


# ============================================================
# [P] Sentence pooling
#
# h: (L, 1024), states for one sentence's tokens.
# c-5 is causal, so h[-1] represents the sentence after its
# available left context has been read.
# ============================================================
def pool_sentence(h):
    return h[-1]


POOL_DIM = pool_sentence(
    torch.zeros(3, FEAT_DIM, device=device)
).numel()


# ============================================================
# [N] Sequence head
#
# Input Z:     (B, S, POOL_DIM)
# lengths:     CPU LongTensor, real sentence count per window
# Output:      (B, S, N_LABELS)
# ============================================================
class SentenceSeqHead(nn.Module):
    def __init__(self, in_dim, n_labels):
        super().__init__()

        self.norm = nn.LayerNorm(in_dim)

        self.lstm = nn.LSTM(
            input_size=in_dim,
            hidden_size=256,
            num_layers=1,
            batch_first=True,
            bidirectional=True,
        )

        self.dropout = nn.Dropout(0.2)
        self.classifier = nn.Linear(512, n_labels)

    def forward(self, Z, lengths):
        Z = self.norm(Z)

        packed = torch.nn.utils.rnn.pack_padded_sequence(
            Z,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False,
        )

        packed_out, _ = self.lstm(packed)

        out, _ = torch.nn.utils.rnn.pad_packed_sequence(
            packed_out,
            batch_first=True,
            total_length=Z.size(1),
        )

        out = self.dropout(out)
        logits = self.classifier(out)

        return logits


head = SentenceSeqHead(
    POOL_DIM, N_LABELS
)

model = SeqRRModel(
    lm,
    pool_sentence,
    head,
).to(device)

count_params(model)


# ============================================================
# Build packed examples
# ============================================================
train_ex = build_packed_examples(
    {d: train[d] for d in TRAIN_DOMAINS},
    **PACK,
)

dev_ex = build_packed_examples(
    dev,
    **PACK,
)

n_pred = sum(
    k is not None
    for ex in train_ex
    for k in ex["keys"]
)

print(
    f"train windows {len(train_ex)} "
    f"({n_pred} predicted sentences) | "
    f"dev windows {len(dev_ex)} | "
    f"longest window "
    f"{max(len(ex['ids']) for ex in train_ex)} tokens"
)


# ============================================================
# Optimizer parameter groups
# ============================================================
head_ids = {
    id(p) for p in model.head.parameters()
}

if isinstance(model.pool, nn.Module):
    head_ids |= {
        id(p) for p in model.pool.parameters()
    }

adapter_params = [
    p for p in model.parameters()
    if p.requires_grad and id(p) not in head_ids
]

head_params = [
    p for p in model.parameters()
    if p.requires_grad and id(p) in head_ids
]

assert adapter_params, "No trainable adapter parameters found"
assert head_params, "No trainable head parameters found"

groups = [
    {"params": adapter_params, "lr": LR},
    {"params": head_params, "lr": HEAD_LR},
]

print(
    f"adapter params "
    f"{sum(p.numel() for p in adapter_params) / 1e6:.3f}M "
    f"at lr {LR} | "
    f"head params "
    f"{sum(p.numel() for p in head_params) / 1e6:.3f}M "
    f"at lr {HEAD_LR}"
)


# ============================================================
# Steps and AdaLoRA rank allocator
# ============================================================
steps_per_epoch = len(
    make_batches(
        train_ex,
        BATCH_SIZE,
        shuffle=False,
    )
)

steps = EPOCHS * steps_per_epoch

assert steps > 0, "No training steps found"

allocator = AdaLoRAAllocator(
    ada_layers,
    ADA_INIT_RANK,
    ADA_TARGET_RANK,
    steps,
    tinit=int(TINIT_FRAC * steps),
    tfinal=int(TFINAL_FRAC * steps),
    delta_t=DELTA_T,
    beta1=BETA1,
    beta2=BETA2,
)

print(
    f"rank budget {allocator.init_budget} -> "
    f"{allocator.target_budget} triplets over "
    f"{len(ada_layers)} layers | {steps} steps, "
    f"no pruning for 1-{allocator.tinit}, "
    f"ranks frozen after {steps - allocator.tfinal}"
)


# ============================================================
# Loss: task loss + AdaLoRA orthogonality regularization
# ============================================================
def adalora_loss(logits, b):
    task_loss = ce_loss(
        logits,
        b["labels"],
        CLASS_WEIGHTS,
    )

    orth_loss = allocator.orth_penalty()

    return task_loss + ORTH_REG * orth_loss


# ============================================================
# Train
# ============================================================
log, best_res = train_model(
    model,
    train_ex,
    dev_ex,
    dev,
    loss_fn=adalora_loss,
    lr=LR,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    weight_decay=WEIGHT_DECAY,
    lr_lambda=warmup_cosine(steps, WARMUP_FRAC),
    param_groups=groups,
    hooks=allocator,
    save_path=f"/kaggle/working/{RUN_NAME}_best_trainable.pt",
)


# ============================================================
# Load best checkpoint
# ============================================================
best_path = f"/kaggle/working/{RUN_NAME}_best_trainable.pt"

state = torch.load(
    best_path,
    map_location=device,
    weights_only=True,
)

model.load_state_dict(
    state,
    strict=False,
)


# ============================================================
# Evaluate on dev
# ============================================================
dev_pred = predict(
    model,
    dev_ex,
    dev,
    BATCH_SIZE,
)

for dom in DOMAINS:
    confusion(dev_pred, dev, dom)

print(
    f"dev inference time "
    f"{time_inference(model, dev_ex, dev, BATCH_SIZE):.1f} s"
)


# ============================================================
# Rank report
# ============================================================
ranks = allocator.rank_pattern()

print(
    "final rank per layer (block x target):\n        "
    + " ".join(f"{t:>9s}" for t in ADA_TARGETS)
)

for bi in range(cfg.n_layers):
    print(
        f"block {bi} "
        + " ".join(
            f"{r:9d}"
            for r in ranks[
                bi * len(ADA_TARGETS):
                (bi + 1) * len(ADA_TARGETS)
            ]
        )
    )

used = sum(
    r * (m.in_features + m.out_features + 1)
    for r, m in zip(ranks, ada_layers)
)

print(
    f"triplets kept {sum(ranks)} / "
    f"{allocator.init_budget} | "
    f"adapter params in use {used / 1e6:.3f}M"
)

with open(
    f"/kaggle/working/{RUN_NAME}_ranks_log.json",
    "w",
) as f:
    json.dump(
        {
            "ranks": dict(zip(ada_names, ranks)),
            "log": log,
        },
        f,
        indent=1,
    )


# ============================================================
# Predict CL + IT test and create submission
# ============================================================
write_submission_packed(
    model,
    PACK,
    BATCH_SIZE,
    best_path=best_path,
)


wrapped 32 linears | AdaLoRA identity-at-init max |diff| = 0.00e+00 (should be ~0)
trainable 6.897M / total 105.31M (6.55%)


CL examples (packed):   0%|          | 0/40 [00:00<?, ?it/s]

IT examples (packed):   0%|          | 0/40 [00:00<?, ?it/s]

CL examples (packed):   0%|          | 0/5 [00:00<?, ?it/s]

IT examples (packed):   0%|          | 0/5 [00:00<?, ?it/s]

train windows 864 (16845 predicted sentences) | dev windows 110 | longest window 1023 tokens
adapter params 4.262M at lr 3e-05 | head params 2.634M at lr 0.0003
rank budget 2560 -> 1696 triplets over 32 layers | 43200 steps, no pruning for 1-4320, ranks frozen after 34560


... (154 outputs left out)

CL examples (packed):   0%|          | 0/5 [00:00<?, ?it/s]

IT examples (packed):   0%|          | 0/5 [00:00<?, ?it/s]

predict:   0%|          | 0/113 [00:00<?, ?it/s]

10 docs, 2197 sentences -> /kaggle/working/submission.zip
